## Multi-Agent Session Example

Join multiple agents to a session and have them return a chain response.

In [1]:
import os

os.environ["LOG_LEVEL"] = "WARNING"
import json
import asyncio
from gai.sessions import SessionManager
from rich.console import Console

console = Console(force_terminal=True)
from gai.messages.typing import MessagePydantic
from gai.sessions.operations.chat import ChatResponder
from gai.lib.tests import make_local_tmp
from gai.messages.dialogue import Dialogue

dialogue = Dialogue(agent_name="User")

# Helper function to create a node that returns mock agent responses

async def create_node(node_name: str, session_mgr: SessionManager, plan):

    async def input_chunks_handler(pydantic: MessagePydantic):
        if pydantic.body.type == "chat.reply":
            # Should never handle reply messages.
            raise ValueError("input_chunks_callback should not process reply messages.")

        # Return a simulated streamer
        from gai.asm.agents import ToolUseAgent
        from gai.lib.config import config_helper
        llm_config = config_helper.get_client_config("sonnet-4")
        agent = ToolUseAgent(
            agent_name=node_name,
            llm_config=llm_config
        )
        
        recap = dialogue.extract_recap()        

        # Time to call chat completion
        async def get_streamer():
            from gai.asm.agents.tool_use_agent import AutoResumeError
            resp=agent.start(user_message=pydantic.body.content,recap=recap)
            content=""
            async for chunk in resp:
                if isinstance(chunk,str):
                    if chunk:
                        content += chunk
                yield chunk
            content+="\n"
            try:
                resp = agent.resume()
                async for chunk in resp:
                    if isinstance(chunk, str):
                        if chunk:
                            content += chunk
                        yield chunk
            except AutoResumeError:
                # conversation is over.
                if not content:
                    content = "My task is completed and I have nothing to resume from."
                dialogue.add_user_message(
                    recipient=agent.fsm.agent_name, content=pydantic.body.content
                )
                dialogue.add_assistant_message(
                    sender=agent.fsm.agent_name, chunk="<eom>", content=content
                )
                return

        return get_streamer()

    async def completed_content_handler(pydantic: MessagePydantic):
        session_mgr.log_message(pydantic)

    node = ChatResponder(node_name=node_name, session_mgr=session_mgr)
    await node.subscribe(
        input_chunks_callback=input_chunks_handler,
        completed_content_callback=completed_content_handler,
    )
    node.plans[plan.dialogue_id] = plan.model_copy()
    return node


# Helper function for displaying the results


def print_title(title):
    console.print(
        f"""\n[bold bright_yellow]{len(title) * "="}\n{title}\n{len(title) * "="}[/bold bright_yellow]\n"""
    )


def show_messages(dialogue):
    print_title("dialogue.messages")
    print("The output below shows the messages saved in dialogue.messages:")
    for message in dialogue.messages:
        print(f"ID: {message.id}")
        print(f"  Sender: {message.header.sender}")
        print(f"  Recipient: {message.header.recipient}")
        print(f"Body:\n{json.dumps(message.body.model_dump(), indent=4)}")
        print("\n" + "-" * 10 + "\n")


In [2]:
import os
import asyncio
from gai.sessions.operations.chat import ChatSender
from rich.console import Console
from gai.lib.tests import make_local_tmp
from gai.sessions import SessionManager

console = Console(force_terminal=True)

# Get the current directory

here = os.getcwd()

# Initialize the dialogue
app_dir = make_local_tmp()
session_mgr = SessionManager(file_path=os.path.join(app_dir, "dialogue.json"))
await session_mgr.start()

## Create plan

from gai.sessions.operations.handshake import HandshakeSender

plan = HandshakeSender.create_plan("""
    User ->> Sara
    Sara ->> Diana
    """)

## Register Sara and wait for User

sara = await create_node("Sara", session_mgr, plan)

## Register Diana and wait for User

diana = await create_node("Diana", session_mgr, plan)

# Create an output message queue for agent response

a_queue = asyncio.Queue()

async def output_chunks_handler(pydantic: MessagePydantic):
    a_queue.put_nowait(pydantic)

# Register the User

user = ChatSender(node_name="User", session_mgr=session_mgr)

await user.subscribe(output_chunks_callback=output_chunks_handler)

async def stream_response(a_queue):
    chunk = await a_queue.get()

    while chunk.body.chunk != "<eom>":
        if chunk.body.chunk_no == 0:
            console.print(f"[bright_green]{chunk.header.sender}[/bright_green] :")
        if isinstance(chunk.body.chunk, str):
            print(chunk.body.chunk, end="", flush=True)
            chunk = await a_queue.get()

    return len(user.plan.steps) - 1 > user.plan.curr_step_no

## START Chain Response

step=await user.chat_send(user_message="Tell me a one paragraph story about a dragon and a knight.", plan=plan)
can_resume = await stream_response(a_queue)
while can_resume:
    await user.next()
    can_resume = await stream_response(a_queue)


Sara :

Hello! I'm Sara, and I'd be happy to tell you a story about a dragon and a knight. Here's a tale for you:

In the mist-shrouded mountains of Eldoria, Sir Gareth approached the ancient dragon Pyraxis not with sword drawn, but with a worn leather satchel containing his grandmother's famous honey cakes. The fearsome beast, whose scales shimmered like molten gold, had been terrorizing the nearby village not out of malice, but from a centuries-old loneliness that had curdled into bitterness. As Gareth sat cross-legged before the dragon and shared his simple offering, speaking of his own losses and dreams, Pyraxis felt something long-forgotten stir within his ancient heart. By sunset, the unlikely pair had forged not a battle, but a friendship that would become legend—the dragon becoming the village's protector, and the knight becoming the keeper of stories that reminded everyone that sometimes the greatest courage lies not in fighting monsters, but in seeing the soul beneath the scales.


Diana :

What a beautiful story, Sara! I love how you turned the traditional dragon-slaying tale on its head. The image of Sir Gareth approaching with honey cakes instead of a sword is so touching, and the idea that Pyraxis was acting out of loneliness rather than evil adds such depth to the story. 

There's something really powerful about the message that "the greatest courage lies not in fighting monsters, but in seeing the soul beneath the scales." It reminds me that so often what we perceive as threatening or dangerous might actually be someone in pain who just needs understanding and connection.

The ending where they become friends and the dragon becomes the village's protector is perfect - it shows how acts of kindness and empathy can transform not just individuals, but entire communities. Thank you for sharing such a heartwarming tale!


In [3]:
show_messages(session_mgr)


=================
dialogue.messages
=================

The output below shows the messages saved in dialogue.messages:
ID: 2dd586b8-5927-4302-98c0-eb3c5e7d20bd
  Sender: User
  Recipient: Sara
Body:
{
    "type": "chat.send",
    "dialogue_id": "00000000-0000-0000-0000-000000000000",
    "round_no": 0,
    "step_no": 0,
    "message_id": "00000000-0000-0000-0000-000000000000.1",
    "content_type": "text",
    "role": "user",
    "content": "Tell me a one paragraph story about a dragon and a knight."
}

----------

ID: bf935436-97a3-4f1a-9f74-bf8c8b0d169f
  Sender: Sara
  Recipient: User
Body:
{
    "type": "chat.reply",
    "dialogue_id": "00000000-0000-0000-0000-000000000000",
    "round_no": 0,
    "step_no": 1,
    "message_id": "00000000-0000-0000-0000-000000000000.3",
    "chunk_no": 18,
    "chunk": "<eom>",
    "content_type": "text",
    "role": "assistant",
    "content": "In the misty highlands of Elderwood, Sir Gareth approached the ancient dragon Pyraxis not with sword drawn, but with a worn leather satchel containing the last